# Solutions — Positional, Keyword, Default, *args, **kwargs

Exercise notebook: `05_functions_and_oop/01_args_kwargs.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Build S3 paths from parts

**Working solution**

In [ ]:
def s3_path(bucket, *parts, filename=None):
    pieces = [p.strip("/") for p in (*parts, filename or "") if p and p.strip("/")]
    return "/".join([f"s3://{bucket.strip('/')}", *pieces])

**Explanation**

Parameters after `*parts` can only be passed by keyword. `(*parts, x)` builds a tuple by unpacking — handy for appending to varargs.

**Complexity:** O(total length).

**Common mistakes**

- Passing the filename positionally — it just becomes another part.

## Exercise 2 — 🟡 Medium · Positional-only and keyword-only parameters

**Working solution**

In [ ]:
def format_event(event_type, /, *, level="INFO", **fields):
    def fmt(v):
        s = str(v)
        return f'"{s}"' if " " in s else s
    parts = [level, event_type] + [f"{k}={fmt(v)}" for k, v in sorted(fields.items())]
    return " ".join(parts)

**Explanation**

`/` marks everything before it positional-only; `*` marks everything after it keyword-only. This lets the function's own parameter names never collide with user-supplied `**fields`.

**Complexity:** O(k log k).

**Common mistakes**

- Without `/`, `format_event('x', event_type='y')` raises 'got multiple values for argument'.

## Exercise 3 — 🟡 Medium · Merge any number of partial records

**Working solution**

In [ ]:
def merge_records(*records, prefer="last", skip_none=True):
    if prefer not in ("first", "last"):
        raise ValueError("prefer must be 'first' or 'last'")
    ordered = records if prefer == "last" else reversed(records)
    merged = {}
    for record in ordered:
        for key, value in record.items():
            if value is None and skip_none:
                continue
            merged[key] = value
    return merged

**Explanation**

`*records` accepts any number of dicts: `merge_records(a, b, c)` or `merge_records(*list_of_dicts)` at the call site.

**Complexity:** O(total keys).

**Common mistakes**

- `{**a, **b}` — can't skip `None` values.
- Mutating the first record as the accumulator.

## Exercise 4 — 🟠 Hard · Call a function with only the config it accepts

**Working solution**

In [ ]:
import inspect

def call_with_config(func, config):
    sig = inspect.signature(func)
    params = sig.parameters.values()
    if any(p.kind is p.VAR_KEYWORD for p in params):
        accepted = dict(config)
        named = [p for p in params if p.kind is not p.VAR_KEYWORD and p.kind is not p.VAR_POSITIONAL]
    else:
        named = [p for p in params if p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD, p.KEYWORD_ONLY)]
        accepted = {p.name: config[p.name] for p in named if p.name in config}
    missing = [p.name for p in named if p.default is p.empty and p.name not in config]
    if missing:
        raise TypeError(f"{func.__name__} missing required config: {missing}")
    positional = [accepted.pop(p.name) for p in named if p.kind is p.POSITIONAL_ONLY and p.name in accepted]
    return func(*positional, **accepted)

**Explanation**

`inspect.signature` exposes each parameter's kind and default. Building `args`/`kwargs` and calling `func(*args, **kwargs)` is dynamic dispatch
without reflection boilerplate. This exact pattern powers dependency injection in pytest and FastAPI.

**Complexity:** O(params + config).

**Common mistakes**

- `func(**config)` → `TypeError: unexpected keyword argument 'env'`.
- Reporting only the first missing parameter.

## Challenge — 🔴 Very Hard · A tiny command dispatcher driven by annotations

**Working solution**

In [ ]:
import inspect
import shlex

def dispatch(command_line, handlers):
    tokens = shlex.split(command_line)
    if not tokens or tokens[0] not in handlers:
        raise ValueError(f"unknown command; available: {sorted(handlers)}")
    handler = handlers[tokens[0]]
    params = inspect.signature(handler).parameters
    positional, options = [], {}
    rest = iter(tokens[1:])
    for token in rest:
        if not token.startswith("--"):
            positional.append(token)
            continue
        name = token[2:].replace("-", "_")
        param = params.get(name)
        if param is None or param.kind not in (param.KEYWORD_ONLY, param.POSITIONAL_OR_KEYWORD):
            raise TypeError(f"unknown option --{token[2:]}")
        if param.annotation is bool:
            options[name] = True
            continue
        try:
            raw = next(rest)
        except StopIteration:
            raise TypeError(f"option --{token[2:]} needs a value") from None
        convert = param.annotation if param.annotation in (int, float, str) else str
        options[name] = convert(raw)
    return handler(*positional, **options)

**Explanation**

This is a miniature `argparse`/`click`/`typer`: annotations aren't just documentation — they're runtime data you can read. Iterating
`rest` with `next(rest)` lets an option consume its value from the same iterator the loop uses.

**Complexity:** O(tokens).

**Common mistakes**

- Splitting on spaces (breaks quoted values).
- Treating `--dry-run` as needing a value.